In [ ]:
import os
import json

TEST_PATH   = '/content/drive/MyDrive/ru.test_adapt.conll'
FILLERS_DIR = '/content/drive/MyDrive/fillers/'
OUTPUT_JSON = '/content/drive/MyDrive/forbidden_fillers.json'

def get_forbidden_fillers(test_path, fillers_dir):
    """
    Читаем test файл → собираем все тексты сущностей →
    сравниваем с филлерами → сохраняем запрещённые
    """
    test_entities = set()
    with open(test_path, 'r', encoding='utf-8') as f:
        content = f.read()

    for block in content.strip().split('\n\n'):
        current = []
        for line in block.strip().split('\n'):
            if line.startswith('#'):
                continue
            parts = line.split('\t')
            if len(parts) != 4:
                continue
            word, bio = parts[1], parts[3]
            if bio.startswith('B-'):
                if current:
                    test_entities.add(' '.join(current).lower())
                current = [word]
            elif bio.startswith('I-') and current:
                current.append(word)
            else:
                if current:
                    test_entities.add(' '.join(current).lower())
                    current = []
        if current:
            test_entities.add(' '.join(current).lower())

    print(f"Сущностей в тесте: {len(test_entities)}")

    forbidden = {}
    for filename in os.listdir(fillers_dir):
        if not filename.endswith('.txt'):
            continue
        name = filename.replace('.txt', '').strip()
        path = os.path.join(fillers_dir, filename)
        with open(path, 'r', encoding='utf-8') as f:
            lines = [l.strip() for l in f.read().split('\n') if l.strip()]

        banned = []
        for line in lines:
            cols = line.split('\t')
            candidates = [cols[0]]
            if len(cols) >= 3:
                candidates.append(cols[2])
            for text in candidates:
                if text.lower() in test_entities:
                    if text not in banned:
                        banned.append(text)

        if banned:
            forbidden[name] = banned
            print(f"  [{name}] запрещено: {len(banned)}")

    return forbidden

forbidden = get_forbidden_fillers(TEST_PATH, FILLERS_DIR)

with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(forbidden, f, ensure_ascii=False, indent=2)

print(f"\nСохранено: {OUTPUT_JSON}")
print(f"Файлов с запрещёнными филлерами: {len(forbidden)}")

In [ ]:
import os
import json
from pymorphy3 import MorphAnalyzer
import random

morph = MorphAnalyzer()

TRAIN_PATH      = '/content/drive/MyDrive/ru.train.conll'
FILLERS_DIR     = '/content/drive/MyDrive/fillers/'
OUTPUT_PATH     = '/content/drive/MyDrive/ru_augmented.train'
FORBIDDEN_PATH  = '/content/drive/MyDrive/forbidden_fillers.json'

PREP_CASE = {
    'в': 'loct', 'во': 'loct', 'на': 'loct', 'о': 'loct', 'об': 'loct', 'при': 'loct',
    'из': 'gent', 'от': 'gent', 'без': 'gent', 'до': 'gent', 'у': 'gent', 'для': 'gent',
    'к': 'datv', 'по': 'datv',
    'с': 'ablt', 'со': 'ablt', 'за': 'ablt', 'над': 'ablt', 'под': 'ablt',
    'про': 'accs', 'через': 'accs',
}

ENT_TO_FILLERS = {
    'album':           ['albums'],
    'artist':          ['artists'],
    'object_name':     ['books', 'games', 'series', 'paintings'],
    'location':        ['cities', 'metro', 'streets', 'districts'],
    'cuisine':         ['cuisine'],
    'served_dish':     ['dishes'],
    'restaurant_name': ['restaurants'],
    'movie_name':      ['movies'],
    'playlist':        ['playlists'],
    'movie_type':      ['movies'],
}

SERVICE_REPLACEMENTS = [
    'Яндекс Музыка',
    'VK Музыка',
]

In [ ]:
def load_all_fillers(fillers_dir, forbidden_path):
    forbidden = {}
    if os.path.exists(forbidden_path):
        with open(forbidden_path, 'r', encoding='utf-8') as f:
            forbidden = json.load(f)
        print(f"  [OK] Запрещённых файлов: {len(forbidden)}")
    else:
        print(f"  [WARN] forbidden_fillers.json не найден")

    all_fillers = {}
    unique_names = set(f for lst in ENT_TO_FILLERS.values() for f in lst)
    for name in unique_names:
        for filename in [name + '.txt', name + ' .txt']:
            path = os.path.join(fillers_dir, filename)
            if os.path.exists(path):
                with open(path, 'r', encoding='utf-8') as f:
                    lines = [l.strip() for l in f.read().split('\n') if l.strip()]
                result = []
                for line in lines:
                    cols = line.split('\t')
                    if len(cols) >= 3:
                        result.append((cols[0], cols[2]))
                    else:
                        result.append(cols[0])
                forbidden_set = set(forbidden.get(name, []))
                result = [
                    item for item in result
                    if (item[0] if isinstance(item, tuple) else item) not in forbidden_set
                ]

                all_fillers[name] = result
                print(f"  [OK] {name}: {len(result)} филлеров (после фильтрации)")
                break
    return all_fillers

In [ ]:
def get_filler_for_type(ent_type, all_fillers):
    filler_names = ENT_TO_FILLERS.get(ent_type)
    if not filler_names:
        return None
    names = filler_names.copy()
    random.shuffle(names)
    for name in names:
        if name in all_fillers and all_fillers[name]:
            filler = random.choice(all_fillers[name])
            return filler[0] if isinstance(filler, tuple) else filler
    return None

In [ ]:
def conjugate(name, case):
    verb_pos = ['VERB', 'INFN', 'PRTS', 'GRND']
    words = name.split()
    parsed = [morph.parse(word) for word in words]
    isverb = list(filter(lambda x: x[0].tag.POS in verb_pos, parsed))
    if isverb:
        return words
    conjugated = []
    for i, pars in enumerate(parsed):
        if pars[0].tag.POS in ['NOUN', 'ADJF', 'PRTF', 'NUMR']:
            try:
                if pars[0].tag.case == case:
                    conjugated.append(words[i])
                    continue
                var = 0
                while var < len(pars) and pars[var].tag.case != 'nomn':
                    var += 1
                if var >= len(pars):
                    var = 0
                inflected = pars[var].inflect({case})
                word = inflected.word if inflected else words[i]
            except:
                word = words[i]
        else:
            word = words[i]
        conjugated.append(word.capitalize() if i == 0 else word.lower())
    return conjugated

In [ ]:
def detect_case(tokens, ent_type):
    b_idx = None
    for i, token in enumerate(tokens):
        if token['bio'] == f'B-{ent_type}':
            b_idx = i
            break
    if b_idx is None:
        return 'nomn'
    if b_idx > 0:
        prev_word = tokens[b_idx - 1]['word'].lower()
        if prev_word in PREP_CASE:
            return PREP_CASE[prev_word]
    last_word = None
    for token in tokens:
        if token['bio'] in (f'B-{ent_type}', f'I-{ent_type}'):
            last_word = token['word']
    if last_word:
        parsed = morph.parse(last_word)
        if parsed and parsed[0].tag.case:
            return parsed[0].tag.case
    return 'nomn'

In [ ]:
def parse_train(train_path):
    with open(train_path, 'r', encoding='utf-8') as f:
        content = f.read()
    sentences = []
    for block in content.strip().split('\n\n'):
        lines = block.strip().split('\n')
        if not lines:
            continue
        meta = {}
        tokens = []
        for line in lines:
            if line.startswith('# id:'):
                meta['id'] = line.replace('# id:', '').strip()
            elif line.startswith('# text:'):
                meta['text'] = line.replace('# text:', '').strip()
            elif line.startswith('# intent:'):
                meta['intent'] = line.replace('# intent:', '').strip()
            elif line.startswith('# slots:'):
                meta['slots'] = line.replace('# slots:', '').strip()
            elif line.startswith('#'):
                continue
            else:
                parts = line.split('\t')
                if len(parts) == 4:
                    tokens.append({
                        'num':    int(parts[0]),
                        'word':   parts[1],
                        'intent': parts[2],
                        'bio':    parts[3]
                    })
        if meta and tokens:
            sentences.append({'meta': meta, 'tokens': tokens})
    print(f"Загружено предложений: {len(sentences)}")
    return sentences

In [ ]:
def replace_entity(tokens, ent_type, all_fillers, case):
    intent = tokens[0]['intent']
    new_tokens = []
    num = 1
    i = 0
    while i < len(tokens):
        bio = tokens[i]['bio']
        if bio == f'B-{ent_type}':
            i += 1
            while i < len(tokens) and tokens[i]['bio'] == f'I-{ent_type}':
                i += 1
            filler_text = get_filler_for_type(ent_type, all_fillers)
            if filler_text is None:
                continue
            new_words = conjugate(filler_text, case)
            for k, word in enumerate(new_words):
                new_tokens.append({
                    'num':    num,
                    'word':   word,
                    'intent': intent,
                    'bio':    f'B-{ent_type}' if k == 0 else f'I-{ent_type}'
                })
                num += 1
        else:
            new_tokens.append({
                'num':    num,
                'word':   tokens[i]['word'],
                'intent': tokens[i]['intent'],
                'bio':    tokens[i]['bio']
            })
            num += 1
            i += 1
    return new_tokens

In [ ]:
def replace_service(tokens):
    intent = tokens[0]['intent']
    new_tokens = []
    num = 1
    i = 0
    while i < len(tokens):
        bio = tokens[i]['bio']
        if bio == 'B-service':
            i += 1
            while i < len(tokens) and tokens[i]['bio'] == 'I-service':
                i += 1
            service = random.choice(SERVICE_REPLACEMENTS)
            for k, word in enumerate(service.split()):
                new_tokens.append({
                    'num':    num,
                    'word':   word,
                    'intent': intent,
                    'bio':    'B-service' if k == 0 else 'I-service'
                })
                num += 1
        else:
            new_tokens.append({
                'num':    num,
                'word':   tokens[i]['word'],
                'intent': tokens[i]['intent'],
                'bio':    tokens[i]['bio']
            })
            num += 1
            i += 1
    return new_tokens

In [ ]:
def recompute_slots(tokens):
    text = ''
    positions = []
    for token in tokens:
        positions.append(len(text))
        text += token['word'] + ' '
    text = text.rstrip()
    slots = []
    i = 0
    while i < len(tokens):
        bio = tokens[i]['bio']
        if bio.startswith('B-'):
            ent_type = bio[2:]
            start = positions[i]
            j = i + 1
            while j < len(tokens) and tokens[j]['bio'] == f'I-{ent_type}':
                j += 1
            end = positions[j-1] + len(tokens[j-1]['word'])
            slots.append(f"{start}:{end}:{ent_type}")
            i = j
        else:
            i += 1
    return text, ', '.join(slots)

In [ ]:
def format_example(meta, new_tokens):
    new_text, new_slots = recompute_slots(new_tokens)
    lines = [
        f"# id: {meta['id']}",
        f"# text: {new_text}",
        f"# intent: {meta['intent']}",
    ]
    if new_slots:
        lines.append(f"# slots: {new_slots}")
    for t in new_tokens:
        lines.append(f"{t['num']}\t{t['word']}\t{t['intent']}\t{t['bio']}")
    return '\n'.join(lines)

In [ ]:
def run_augmentation():
    print("=== Загрузка филлеров ===")
    all_fillers = load_all_fillers(FILLERS_DIR, FORBIDDEN_PATH)

    print("\n=== Парсинг train ===")
    sentences = parse_train(TRAIN_PATH)

    augmented_by_id = {}
    skipped = 0

    for sent in sentences:
        tokens = sent['tokens']

        ent_types_in_sent = []
        for token in tokens:
            if token['bio'].startswith('B-'):
                ent_type = token['bio'][2:]
                if ent_type in ENT_TO_FILLERS and ent_type not in ent_types_in_sent:
                    ent_types_in_sent.append(ent_type)

        has_service = any(t['bio'] == 'B-service' for t in tokens)

        if not ent_types_in_sent and not has_service:
            skipped += 1
            continue

        new_tokens = [t.copy() for t in tokens]
        changed = False

        for ent_type in ent_types_in_sent:
            case = detect_case(new_tokens, ent_type)
            new_tokens = replace_entity(new_tokens, ent_type, all_fillers, case)
            changed = True

        if has_service:
            new_tokens = replace_service(new_tokens)
            changed = True

        if changed:
            augmented_by_id[sent['meta']['id']] = format_example(sent['meta'], new_tokens)

    with open(TRAIN_PATH, 'r', encoding='utf-8') as f:
        original_train = f.read().strip()

    original_blocks = original_train.split('\n\n')
    result_blocks = []

    for block in original_blocks:
        block_id = None
        for line in block.split('\n'):
            if line.startswith('# id:'):
                block_id = line.replace('# id:', '').strip()
                break
        if block_id and block_id in augmented_by_id:
            result_blocks.append(augmented_by_id[block_id])
        else:
            result_blocks.append(block)

    with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
        for i, block in enumerate(result_blocks, start=1):
            lines = block.split('\n')
            lines[0] = f'# id: train_{i}'
            f.write('\n'.join(lines))
            f.write('\n\n')

    print(f"\nГотово!")
    print(f"  Всего предложений:    {len(original_blocks)}")
    print(f"  Заменено:             {len(augmented_by_id)}")
    print(f"  Без изменений:        {skipped}")
    print(f"  Итого в файле:        {len(result_blocks)}")
    print(f"  Сохранено:            {OUTPUT_PATH}")

run_augmentation()